# Convenience Functions

Lowercase scoring functions are the direct interface for one plan. Pass the source first, the
assignment second, and then name the columns required by the score.

In [ ]:
import geopandas as gpd
import networkx as nx
import pandas as pd
from shapely.geometry import box

import gerrytools.scoring as gs

units = gpd.GeoDataFrame(
    {
        "district": [0, 0, 1, 1],
        "population": [90, 110, 95, 105],
        "vap": [70, 80, 75, 85],
        "bvap": [20, 30, 35, 45],
        "democratic": [60, 55, 40, 45],
        "republican": [40, 45, 60, 55],
    },
    geometry=[
        box(0, 1, 1, 2),
        box(1, 1, 2, 2),
        box(0, 0, 1, 1),
        box(1, 0, 2, 1),
    ],
    crs="EPSG:3857",
)
units

## Basic Convenience functions

A GeoDataFrame assignment can be the name of a column. One requested tally returns a Series;
several requested tallies return a DataFrame.

In [ ]:
district_totals = gs.tally(
    units,
    "district",
    columns=["population", "vap", "bvap"],
)
district_totals

The other convenience functions follow the same pattern. They return a scalar, Series, or
DataFrame with the natural labels for that score.

In [ ]:
bvap_share = gs.demographic_shares(
    units,
    "district",
    subgroup_population_attr="bvap",
    total_population_attr="vap",
)
compactness = gs.polsby_popper(units, "district")
democratic_seats = gs.seats(
    units,
    "district",
    party_vote_attr="democratic",
    opposition_vote_attr="republican",
)

pd.DataFrame(
    {
        "BVAP share": bvap_share,
        "Polsby-Popper": compactness,
    }
).assign(democratic_seats=democratic_seats)

## Use graph assignments for topological scores

Graph-backed functions accept a node-to-district mapping or a sequence in graph-node order.

In [ ]:
graph = nx.convert_node_labels_to_integers(nx.grid_2d_graph(2, 2))
assignment = units["district"].tolist()
gs.cut_edges(graph, assignment)

Use [PlanEvaluator](plan_evaluator.ipynb) when several scores should share prepared resources
or when more than one plan needs to be evaluated. Use
[Working with GerryChain](basic.ipynb) when the source is a partition or Markov chain.